# Contraction Mapping Visualizer

Chapter 15: Bellman Equations and Operators · *Mathematics for AI and Machine Learning* by Fuheng Wu

See geometric error decay under repeated Bellman operator applications.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-15/contraction/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's defaults: discount gamma, starting distance d0, number of steps K
gamma, d0, K = 0.85, 2.0, 35

# A random 5-state MDP with 3 actions
S, A = 5, 3
P = torch.rand(A, S, S, dtype=torch.float64)
P = P / P.sum(-1, keepdim=True)
R = torch.rand(A, S, dtype=torch.float64)


def T(V):
    """Bellman optimality operator: max_a [R(a, s) + gamma P(a) V]."""
    return (R + gamma * P @ V).max(dim=0).values


# ── Contraction: ||T V - T U||_inf <= gamma ||V - U||_inf for ANY V, U ───────
worst = 0.0
for _ in range(10000):
    V, U = torch.randn(S, dtype=torch.float64) * 5, torch.randn(S, dtype=torch.float64) * 5
    worst = max(worst, ((T(V) - T(U)).abs().max() / (V - U).abs().max()).item())
print(f"largest ratio ||TV - TU|| / ||V - U|| over 10000 random pairs = {worst:.4f} <= gamma = {gamma}")

# ── Two value functions d0 apart: the gap shrinks at least like gamma^k ─────
# A constant shift is the worst case: T(V + d) = T(V) + gamma d, so it meets the bound exactly
V = torch.zeros(S, dtype=torch.float64)
U = V + d0
print("\n  k   ||V_k - U_k||    bound d0 gamma^k")
for k in range(K + 1):
    if k in (0, 1, 2, 5, 10, 20, K):
        print(f"{k:3d}   {(V - U).abs().max().item():.6f}       {d0 * gamma ** k:.6f}")
    V, U = T(V), T(U)

# A non-constant difference usually shrinks faster than the bound
V, U = torch.zeros(S, dtype=torch.float64), d0 * torch.rand(S, dtype=torch.float64)
for _ in range(10):
    V, U = T(V), T(U)
print(f"random start, after 10 steps: {(V - U).abs().max().item():.6f} <= bound {d0 * gamma ** 10:.6f}")

# Banach: both sequences meet at the unique fixed point V*
for _ in range(500):
    V = T(V)
print("\nfixed point V* =", [round(v, 4) for v in V.tolist()], " ||T V* - V*|| =", (T(V) - V).abs().max().item())

largest ratio ||TV - TU|| / ||V - U|| over 10000 random pairs = 0.7789 <= gamma = 0.85

  k   ||V_k - U_k||    bound d0 gamma^k
  0   2.000000       2.000000
  1   1.700000       1.700000
  2   1.445000       1.445000
  5   0.887411       0.887411
 10   0.393749       0.393749
 20   0.077519       0.077519
 35   0.006772       0.006772
random start, after 10 steps: 0.188391 <= bound 0.393749

fixed point V* = [3.2692, 3.3486, 3.1366, 3.2566, 3.1687]  ||T V* - V*|| = 0.0
